In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "bc1_well" # or None
random_seed = 42

condition_keys = "cytokine" # 数据集perturbation所对应的obs列名
dataset_name = "PBMC_all_hvg2000_gemini_test1"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"

cov_config = {
    "donor": {
        "type": "categorical",
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
    # "cell_type": {
    #     "type": "categorical",
    #     "control_ot": "global",
    #     "perturbed_ot": "global",
    #     "use_in_model": True,
    #     "model_source": "control",
    #     "contain_in_condition": False,
    #     "condition_source": None,
    # },
}


if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"


#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
condition_rep_dict = pd.read_pickle("./data/processed/condition_embedding_cytokine_PBMC_gemini_test1.pkl")
condition_rep_dict = {
    k: (v["embedding"] if isinstance(v, dict) and "embedding" in v else None)
    for k, v in condition_rep_dict.items()
}

In [6]:
filePath = './data/raw/PBMC_all_hvg2000.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

AnnData object with n_obs × n_vars = 9697974 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config'
    layers: 'counts'


In [7]:
adata.obs[control_key] = (adata.obs[condition_keys] == "PBS")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    9068273
True      629701
Name: count, dtype: int64


## splitting

In [8]:
#adata = adata[~adata.obs[condition_keys].isin(["LT-alpha2-beta1","IFN-lambda2","IFN-lambda3","IL-18Ra","LT-alpha1-beta2"])]

In [9]:
rng = np.random.default_rng(random_seed)
test_ratio = "cellflow"
donor_num = 0
condition_list = list(condition_list)
zero_shot = True

# 建议先统一 condition 表示，避免 condition_keys 是多列时 zeroshot 分支出错
if isinstance(condition_keys, str):
    condition_series = adata.obs[condition_keys].astype(str)
else:
    condition_series = adata.obs[condition_keys].astype(str).agg("_".join, axis=1)

if not zero_shot:
    # 分层抽样，先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = condition_series.loc[pert_mask].values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()

    adata_train.uns["normalized_m"] = 1 / (1 - test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1

else:
    # --------------------------------------------------
    # zero-shot: 按 condition 划 test
    # 但从 donor 中随机抽 donor_num 个，
    # 把这些 donor 在 test_condition 下的样本保留到训练集
    # --------------------------------------------------
    if test_ratio == "cellflow":
        test_condition = ["4-1BBL", "ADSF", "APRIL", "BAFF", "C5a", "IFN-beta", "IL-13", "IL-15", "Noggin", "OSM", "OX40L", "IFN-epsilon"]
    else:
        n_test = max(1, int(len(condition_list) * test_ratio))
        test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    train_condition = [c for c in condition_list if c not in test_condition]

    # donor 抽样
    all_donors = sorted(adata.obs["donor"].dropna().unique().tolist())
    donor_num = min(donor_num, len(all_donors))
    selected_donors = rng.choice(all_donors, size=donor_num, replace=False).tolist()

    print("test_condition:", test_condition)
    print("train_condition:", train_condition)
    print("selected_donors_for_test_in_train:", selected_donors)

    control_mask = adata.obs[control_key] == True
    pert_mask = ~control_mask

    train_condition_mask = condition_series.isin(train_condition)
    test_condition_mask = condition_series.isin(test_condition)
    selected_donor_mask = adata.obs["donor"].isin(selected_donors)

    # 训练集：
    # 1) 所有 train_condition
    # 2) test_condition 里属于 selected_donors 的样本
    train_mask = pert_mask & (
        train_condition_mask |
        (test_condition_mask & selected_donor_mask)
    )

    # 测试集：
    # test_condition 里不属于 selected_donors 的样本
    test_mask = pert_mask & test_condition_mask & (~selected_donor_mask)

    adata_control = adata[control_mask].copy()
    adata_train = adata[train_mask].copy()
    adata_test = adata[test_mask].copy()

    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1

    print("train cells:", adata_train.n_obs)
    print("test cells:", adata_test.n_obs)
    print("control cells:", adata_control.n_obs)


test_condition: ['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'C5a', 'IFN-beta', 'IL-13', 'IL-15', 'Noggin', 'OSM', 'OX40L', 'IFN-epsilon']
train_condition: ['Megalin', 'CD27L', 'CD30L', 'CD40L', 'CT-1', 'Decorin', 'EGF', 'EPO', 'FGF-beta', 'FLT3L', 'FasL', 'G-CSF', 'GDNF', 'GITRL', 'GM-CSF', 'HGF', 'IFN-alpha1', 'IFN-gamma', 'IFN-lambda1', 'IFN-lambda2', 'IFN-lambda3', 'IFN-omega', 'IGF-1', 'IL-1-alpha', 'IL-1-beta', 'IL-10', 'IL-11', 'IL-12', 'IL-16', 'IL-17A', 'IL-17B', 'IL-17C', 'IL-17D', 'IL-17E', 'IL-17F', 'IL-18Ra', 'IL-19', 'IL-1Ra', 'IL-2', 'IL-20', 'IL-21', 'IL-22', 'IL-23', 'IL-24', 'IL-26', 'IL-27', 'IL-3', 'IL-31', 'IL-32-beta', 'IL-33', 'IL-34', 'IL-35', 'IL-36-alpha', 'IL-36Ra', 'IL-4', 'IL-5', 'IL-6', 'IL-7', 'IL-8', 'IL-9', 'LIF', 'LIGHT', 'LT-alpha1-beta2', 'LT-alpha2-beta1', 'Leptin', 'M-CSF', 'PRL', 'PSPN', 'RANKL', 'SCF', 'LAP-TGF-beta1', 'TL1A', 'TNF-alpha', 'TPO', 'TRAIL', 'TSLP', 'TWEAK', 'VEGF']
selected_donors_for_test_in_train: []
train cells: 7863725
test cells: 12045

In [10]:
del adata

## latent embedding

In [11]:
n_comps = 100
n_hidden = 1024
n_layers = 2

model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [12]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [13]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

[8.020406   4.2945304  3.5864155  2.524642   2.0648465  1.8617562
 1.8121787  1.7528948  1.5942163  1.5666741  1.4554176  1.3726728
 1.2928425  1.2245581  1.2188879  1.1852355  1.1451678  1.1276438
 1.1001124  1.0801299  1.0556259  1.0617256  1.0483427  1.0148853
 1.0016797  0.9921702  0.9752628  0.96864307 0.9571388  0.949065
 0.95419633 0.93640333 0.9221589  0.91974384 0.9185755  0.91002005
 0.8978755  0.893635   0.8889953  0.8818143  0.87676924 0.86757684
 0.86256015 0.8568511  0.8532293  0.8427597  0.81967306 0.83485556
 0.8210523  0.818056   0.81399333 0.81601727 0.79394156 0.79392165
 0.79087263 0.79332775 0.7758844  0.7783446  0.7683095  0.7704065
 0.7702422  0.7599584  0.75810325 0.7529815  0.75213134 0.7482559
 0.7480525  0.73971176 0.7368833  0.73451287 0.73399115 0.72695845
 0.7273793  0.7261778  0.7162451  0.7175988  0.71560764 0.7080513
 0.70739067 0.6967943  0.6978867  0.69918674 0.6927713  0.6871432
 0.69114774 0.69049406 0.6879045  0.6811691  0.6764938  0.67164373
 0.67

In [14]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{donor_num}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [15]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/PBMC_all_hvg2000_gemini_test1_42_cellflow_0_True_X_pca_100_None
AnnData object with n_obs × n_vars = 629701 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot', 'is_control', 'donor_idx', 'condition_combined'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config', 'normalized_m', 'pca', 'global_rulebook'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 7863725 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_coun

In [16]:
adata_control.uns

{'hvg': {'flavor': 'seurat'},
 'log1p': {},
 'cov_config': {'donor': {'type': 'categorical',
   'control_ot': 'groupwise',
   'perturbed_ot': 'groupwise',
   'use_in_model': True,
   'model_source': 'both',
   'contain_in_condition': True,
   'condition_source': 'both'}},
 'normalized_m': 1,
 'pca': {'params': {'zero_center': False,
   'use_highly_variable': False,
   'mask_var': None,
   'layer': 'X_centered'},
  'variance': array([60.37474   , 17.65518   , 13.147407  ,  6.1663866 ,  4.089412  ,
          3.44389   ,  3.1534998 ,  2.9912467 ,  2.5552692 ,  2.380915  ,
          2.106113  ,  1.9618766 ,  1.6947701 ,  1.5188184 ,  1.4999273 ,
          1.3950701 ,  1.3013936 ,  1.272645  ,  1.2044582 ,  1.1850367 ,
          1.1220664 ,  1.1123909 ,  1.0661187 ,  1.0325867 ,  1.0002056 ,
          0.9933486 ,  0.9563506 ,  0.94214743,  0.91826624,  0.9033768 ,
          0.8933305 ,  0.8804797 ,  0.86148965,  0.8514795 ,  0.8434167 ,
          0.81977046,  0.80640966,  0.8011752 ,  0.796

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()